Collaborative Recommendation System 
- Item Based

In [39]:
import pandas as pd

#Loading movie dataset
movies=pd.read_csv("../datasets/movies.csv")
movies.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


Loading ratings dataset

In [40]:
ratings=pd.read_csv("../datasets/ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Data Exploration and Understanding

In [41]:
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [42]:
ratings.duplicated().sum()

np.int64(0)

In [43]:
ratings['movieId'].unique()

array([     1,      3,      6, ..., 160836, 163937, 163981], shape=(9724,))

In [44]:
ratings['rating'].value_counts().sort_index()

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [45]:
movies.duplicated().sum()

np.int64(0)

In [46]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [47]:
movies.columns

Index(['movieId', 'title', 'genres'], dtype='str')

Movie genre distribution

In [48]:
with pd.option_context('display.max_rows',None):
    print(movies['genres'].value_counts())

genres
Drama                                                                            1053
Comedy                                                                            946
Comedy|Drama                                                                      435
Comedy|Romance                                                                    363
Drama|Romance                                                                     349
Documentary                                                                       339
Comedy|Drama|Romance                                                              276
Drama|Thriller                                                                    168
Horror                                                                            167
Horror|Thriller                                                                   135
Crime|Drama                                                                       134
Crime|Drama|Thriller                           

Data cleaning and preprocessing

In [49]:
movies['genres']=movies['genres'].replace('(no genres listed)','')

Merge Movies and Ratings Dataset

In [50]:
# Merge movies and ratings datasets using movieId
movie_ratings=pd.merge(ratings,movies,on='movieId',how='inner')
movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


Movie-User Interaction Matrix

In [51]:
movie_item=ratings.pivot_table(index='movieId',columns='userId',values='rating')
print(movie_item)
print("Movie Item Matrix Shape:",movie_item.shape)

userId   1    2    3    4    5    6    7    8    9    10   ...  601  602  603  \
movieId                                                    ...                  
1        4.0  NaN  NaN  NaN  4.0  NaN  4.5  NaN  NaN  NaN  ...  4.0  NaN  4.0   
2        NaN  NaN  NaN  NaN  NaN  4.0  NaN  4.0  NaN  NaN  ...  NaN  4.0  NaN   
3        4.0  NaN  NaN  NaN  NaN  5.0  NaN  NaN  NaN  NaN  ...  NaN  NaN  NaN   
4        NaN  NaN  NaN  NaN  NaN  3.0  NaN  NaN  NaN  NaN  ...  NaN  NaN  NaN   
5        NaN  NaN  NaN  NaN  NaN  5.0  NaN  NaN  NaN  NaN  ...  NaN  NaN  NaN   
...      ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
193581   NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN  NaN  NaN   
193583   NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN  NaN  NaN   
193585   NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN  NaN  NaN   
193587   NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN  NaN  NaN   
193609   NaN  NaN  NaN  NaN 

Calculate Matrix Density and Sparsity

In [52]:
# Check which movie ratings are available
movie_item.notna()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,True,False,False,False,True,False,True,False,False,False,...,True,False,True,True,True,True,True,True,True,True
2,False,False,False,False,False,True,False,True,False,False,...,False,True,False,True,True,False,False,True,False,False
3,True,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,True,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
193583,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
193585,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [53]:
# Count available ratings for each movie
movie_item.notna().sum()

userId
1       232
2        29
3        39
4       216
5        44
       ... 
606    1115
607     187
608     831
609      37
610    1302
Length: 610, dtype: int64

In [54]:
# Calculate total number of actual ratings
total_rated_movie=movie_item.notna().sum().sum()
total_rated_movie

np.int64(100836)

In [55]:
# Calculate total number of possible movie-user combinations
total_cell=movie_item.shape[0]*movie_item.shape[1]
total_cell

5931640

In [56]:
# Calculate matrix density
density=total_rated_movie/total_cell
print("Density: ",density)

Density:  0.016999683055613623


In [57]:
# Calculate matrix sparsity
sparsity=1-density
print("Sparsity: {:.2%}".format(sparsity))

Sparsity: 98.30%


Average Rating

In [58]:
# axis=1 calculates the average across users in each movie row
movie_mean=movie_item.mean(axis=1)
movie_mean.head()

movieId
1    3.920930
2    3.431818
3    3.259615
4    2.357143
5    3.071429
dtype: float64

Mean Centering

In [59]:
# Subtract each movie's average rating from its actual ratings
# Fill missing ratings with 0 for similarity calculation
movie_item_centered=movie_item.sub(movie_mean,axis=0).fillna(0) 
movie_item_centered.iloc[:5,:5]

userId,1,2,3,4,5
movieId,,,,,
1,0.079070,0.0,0.0,0.0,0.07907
2,0.000000,0.0,0.0,0.0,0.00000
3,0.740385,0.0,0.0,0.0,0.00000
4,0.000000,0.0,0.0,0.0,0.00000
5,0.000000,0.0,0.0,0.0,0.00000


In [60]:
# Convert mean-centered DataFrame into NumPy array
centered_values=movie_item_centered.values
centered_values

array([[ 0.07906977,  0.        ,  0.        , ..., -1.42093023,
        -0.92093023,  1.07906977],
       [ 0.        ,  0.        ,  0.        , ..., -1.43181818,
         0.        ,  0.        ],
       [ 0.74038462,  0.        ,  0.        , ..., -1.25961538,
         0.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(9724, 610))

Rated movie=1, not rated=0

In [61]:
rated_mask = movie_item.notna().values.astype(float)
rated_mask

array([[1., 0., 0., ..., 1., 1., 1.],
       [0., 0., 0., ..., 1., 0., 0.],
       [1., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(9724, 610))

Extract User and Movie IDs

In [62]:
user_ids=movie_item.columns
print(user_ids)
movie_ids=movie_item.index
print(movie_ids)

Index([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,
       ...
       601, 602, 603, 604, 605, 606, 607, 608, 609, 610],
      dtype='int64', name='userId', length=610)
Index([     1,      2,      3,      4,      5,      6,      7,      8,      9,
           10,
       ...
       193565, 193567, 193571, 193573, 193579, 193581, 193583, 193585, 193587,
       193609],
      dtype='int64', name='movieId', length=9724)


Calculate Item-Item Similarity

In [63]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Calculate cosine similarity between movies
# Each row represents a movie, so similarity is calculated between movies
movie_similarity=cosine_similarity(movie_item_centered)

# Set diagonal values to zero to exclude self-similarity
np.fill_diagonal(movie_similarity,0)
print("Similarity Shape: ",movie_similarity.shape)

Similarity Shape:  (9724, 9724)


Movie Similarity Matrix

In [64]:
#movie similarity array into a DataFrame
movie_sim_df=pd.DataFrame(movie_similarity,index=movie_ids,columns=movie_ids)
movie_sim_df.iloc[:5,:5]

movieId,1,2,3,4,5
movieId,,,,,
1,0.000000,0.139649,0.113850,0.032658,0.076230
2,0.139649,0.000000,0.187303,-0.016897,0.154411
3,0.113850,0.187303,0.000000,0.071153,0.220868
4,0.032658,-0.016897,0.071153,0.000000,0.128339
5,0.076230,0.154411,0.220868,0.128339,0.000000


Similar Movies

In [65]:
target_movie=1
movie_sim_df[target_movie].sort_values(ascending=False).head()

movieId
3114    0.318666
6377    0.318374
588     0.316544
8961    0.276687
1367    0.267269
Name: 1, dtype: float64

In [66]:
# Identify the most similar movie
similar_movie=movie_sim_df[target_movie].idxmax()
print("Most similar movie:", similar_movie)

Most similar movie: 3114


Common Users Between Movies

In [67]:
common = movie_ratings[movie_ratings['movieId'].isin([target_movie, similar_movie])]

# User-movie rating table and users who rated both movies
common = common.pivot_table(index='userId', columns='title', values='rating').dropna()
print("Users who rated both:", len(common))
common.head(10)

Users who rated both: 81


title,Toy Story (1995),Toy Story 2 (1999)
userId,,
7,4.5,4.5
19,4.0,2.0
21,3.5,3.5
27,3.0,4.0
45,4.0,5.0
50,3.0,3.0
57,5.0,4.0
64,4.0,4.0
68,2.5,3.0


Predict Movie Ratings Using Item-Based Collaborative Filtering

In [70]:
def predict_for_user(user_id,k=50,min_neighbors=3):
    # Position of the selected user
    u=user_ids.get_loc(user_id)
    
    # Find movies already rated by the selected user
    rated_movies=np.where(rated_mask[:u]==1)[0]
    
    # Initialize predictions for all movies
    predictions = np.full(len(movie_ids), np.nan)

    # Predict ratings for each movie
    for i in range(len(movie_ids)):
        # Calculate similarity between the target movie
        # and movies already rated by the user
        sims = movie_similarity[i, rated_movies]
        
        # Keep only positively similar movies
        valid = sims > 0
        
        neighbor_indices = rated_movies[valid]
        sims = sims[valid]

        # Select the top K similar movies
        top = np.argsort(sims)[::-1][:k]

        neighbor_indices = neighbor_indices[top]
        w = sims[top]

        # Require a minimum number of similar rated movies
        if len(neighbor_indices) < min_neighbors:
            continue

        # Calculate weighted sum of centered ratings
        num = w @ centered_values[neighbor_indices, u]

        # Calculate sum of absolute similarity scores
        den = np.abs(w).sum()

        # Calculate predicted rating
        predictions[i] = movie_mean.iloc[i] + num / den

    # Return predicted ratings as a Pandas Series
    return pd.Series(predictions, index=movie_ids)

Movie Recommendations

In [76]:
#Predicted ratings for user 1
predictions = predict_for_user(1)

# Find movies that user 1 has not watched
unwatched = movie_item[1].isna()

# Select predictions for unwatched movies
recommendations = predictions[unwatched]

# Sort predictions and select the top 10 movies
top_recommendations=recommendations.sort_values(ascending=False).head(10)
top_recommendations

movieId
2    NaN
4    NaN
5    NaN
7    NaN
8    NaN
9    NaN
10   NaN
11   NaN
12   NaN
13   NaN
dtype: float64

Display movie titles

In [79]:
recommended_movies=pd.DataFrame({'movieId':top_recommendations.index, 'predicted_rating': top_recommendations.values})

#Movie Titles
recommended_movies=pd.merge(
    recommended_movies,
    movies[['movieId','title']],
    on='movieId',
    how='left'
)

recommended_movies[['movieId', 'title', 'predicted_rating']]

,movieId,title,predicted_rating
0,2,Jumanji (1995),NaN
1,4,Waiting to Exhale (1995),NaN
2,5,Father of the Bride Part II (1995),NaN
3,7,Sabrina (1995),NaN
4,8,Tom and Huck (1995),NaN
5,9,Sudden Death (1995),NaN
6,10,GoldenEye (1995),NaN
7,11,"American President, The (1995)",NaN
8,12,Dracula: Dead and Loving It (1995),NaN
9,13,Balto (1995),NaN
